In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install gensim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = '/content/drive/MyDrive/Colab_Objects/data/unisa_undergrad_qualif_2024_12_03_08_21_32.csv'
df = pd.read_csv(file_path)
df.head()

,name,code,saqa_id,dep,aps,faculty,nqf,credits,desc,comments,school,url
0,Higher Certificate in Banking (98225),98225,84286,NaN,15,Economic and Management Sciences,5,120,Purpose statement:The primary purpose of the q...,\nRules:\nStudents who have already passed BAN...,NaN,https://www.unisa.ac.za/sites/corporate/defaul...
1,Higher Certificate in Economic and Management ...,98237,90677,NaN,15,Economic and Management Sciences,5,120,Purpose statement:The purpose of this qualific...,\nRules:Students should note that during the c...,NaN,https://www.unisa.ac.za/sites/corporate/defaul...
2,Higher Certificate in Marketing (98229),98229,84766,NaN,15,Economic and Management Sciences,5,120,Purpose statement:The primary purpose of this ...,\nRules:\nThe College has replaced EUP1501 wit...,NaN,https://www.unisa.ac.za/sites/corporate/defaul...
3,Higher Certificate in Retailing (90014),90014,94791,NaN,15,Economic and Management Sciences,5,120,Purpose statement:The purpose of the qualifica...,\nRules:\nStudents who have already passed MNM...,NaN,https://www.unisa.ac.za/sites/corporate/defaul...
4,Higher Certificate in Supervisory Management (...,90015,94630,NaN,15,Economic and Management Sciences,5,120,Purpose statement:The purpose of this qualific...,\nRules:\nStudents should note that during the...,NaN,https://www.unisa.ac.za/sites/corporate/defaul...


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    # Tokenize the text
    words = word_tokenize(text)
    # Get the list of English stopwords
    stop_words = set(stopwords.words('english'))
    # Remove stopwords
    filtered_words = [word for word in words if word.lower() not in stop_words]
    # Join the filtered words back into a string
    text = ' '.join(filtered_words)
    comments_words = ['purpose','may','required','skills','students','level','subject','internet','statement','rules','qualification']
    for word in comments_words:
        text = text.replace(word,'')
    text = re.sub(r'\s{2,}', ' ', text)
    text = text.strip()
    clean_text = text
    return clean_text

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
features_df = df.loc[:,['desc','comments','code','name','faculty']]
features_df['features'] = features_df['desc']+' '+features_df['comments']+' '+features_df['name']
features_df.head()

,desc,comments,code,name,faculty,features
0,Purpose statement:The primary purpose of the q...,\nRules:\nStudents who have already passed BAN...,98225,Higher Certificate in Banking (98225),Economic and Management Sciences,Purpose statement:The primary purpose of the q...
1,Purpose statement:The purpose of this qualific...,\nRules:Students should note that during the c...,98237,Higher Certificate in Economic and Management ...,Economic and Management Sciences,Purpose statement:The purpose of this qualific...
2,Purpose statement:The primary purpose of this ...,\nRules:\nThe College has replaced EUP1501 wit...,98229,Higher Certificate in Marketing (98229),Economic and Management Sciences,Purpose statement:The primary purpose of this ...
3,Purpose statement:The purpose of the qualifica...,\nRules:\nStudents who have already passed MNM...,90014,Higher Certificate in Retailing (90014),Economic and Management Sciences,Purpose statement:The purpose of the qualifica...
4,Purpose statement:The purpose of this qualific...,\nRules:\nStudents should note that during the...,90015,Higher Certificate in Supervisory Management (...,Economic and Management Sciences,Purpose statement:The purpose of this qualific...


In [ ]:
features_df['features'] = features_df['features'].apply(preprocess)
features_df[['features','faculty','code']].head()

,features,faculty,code
0,primary closely tied rationale introduce learn...,Economic and Management Sciences,98225
1,prepare learners comply minimum statutory inst...,Economic and Management Sciences,98237
2,primary closely tied rationale serves provide ...,Economic and Management Sciences,98229
3,equip learners knowledge enter retailing envir...,Economic and Management Sciences,90014
4,prepare learners operate perform operational o...,Economic and Management Sciences,90015


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# Fit Label encoder and return encoded labels
#y = y_balanced #features_df['faculty']
#X = X_balanced #features_df['features']
y = features_df['faculty']
X = features_df['features']
#type_labels = list(le.classes_)

<h3>TFIDF Learner</h3>

In [ ]:
import pickle

# Define file names based on timestamp
tfidf_results_filename = "/content/drive/MyDrive/Colab_Objects/output/tfidf_df_results_2025-07-17_19-48-30.pkl"
tfidf_best_model_filename = "/content/drive/MyDrive/Colab_Objects/output/tfidf_best_per_model_2025-07-17_19-48-30.pkl"
tfidf_external_metrics_filename = "/content/drive/MyDrive/Colab_Objects/output/tfidf_external_metrics_2025-07-17_19-48-30.pkl"

# Load df_results
with open(tfidf_results_filename, 'rb') as f:
    tfidf_df_results = pickle.load(f)

# Load best_per_model
with open(tfidf_best_model_filename, 'rb') as f:
    tfidf_best_per_model = pickle.load(f)

# Load df_external_metrics
with open(tfidf_external_metrics_filename, 'rb') as f:
    tfidf_df_external_metrics = pickle.load(f)

# Display the loaded objects
print("\n======= [TFIDF] Final Averaged Results Across All Folds =======")
print(tfidf_df_results)

print("\n======= [TFIDF] External Test Set Metrics =======")
print(tfidf_df_external_metrics)

print("\n [TFIDF] Best fold per classifier (F1-score):")
for name, entry in tfidf_best_per_model.items():
    print(f"{name}: {entry}")



======= [TFIDF] Final Averaged Results Across All Folds =======
   Accuracy  Precision    Recall  F1-Score                Model
0  0.913964   0.907591  0.780256  0.812546  Logistic Regression
1  0.949807   0.949767  0.904883  0.916940           Linear SVM
2  0.942616   0.908212  0.864951  0.875727        Random Forest
3  0.924678   0.868235  0.859622  0.850694        Decision Tree
4  0.964109   0.967782  0.938629  0.947504       MLP Classifier

======= [TFIDF] External Test Set Metrics =======
                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.416667   0.455128  0.347712  0.311361
1           Linear SVM  0.500000   0.427249  0.428431  0.367778
2        Random Forest  0.166667   0.190476  0.222222  0.125000
3        Decision Tree  0.444444   0.533333  0.485294  0.430556
4       MLP Classifier  0.472222   0.355556  0.359150  0.313735

 [TFIDF] Best fold per classifier (F1-score):
Logistic Regression: {'f1': 0.8425555859943715, 'model': LogisticRegr

<h3>Word2Vec Learner</h3>

In [ ]:
import gensim.downloader
gensim_pretrains = list(gensim.downloader.info()['models'].keys())
model_i = 4 #ruscorpora(deja), glove-twitter-25 (deja), word2vec-google-news-300 (deja)
pretrain_model_name = gensim_pretrains[model_i-1]
word2vec_model = gensim.downloader.load(pretrain_model_name)

In [ ]:
word2vec_model

In [ ]:
import numpy as np
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.base import BaseEstimator, TransformerMixin

class Word2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.vector_size = word2vec_model.vector_size

    def preprocess(self, text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z]+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        words = word_tokenize(text)
        stop_words = set(stopwords.words('english'))
        filtered_words = [word for word in words if word not in stop_words]
        comments_words = ['purpose','may','required','skills','students','level',
                          'subject','internet','statement','rules','qualification']
        filtered_words = [word for word in filtered_words if word not in comments_words]
        return filtered_words

    def transform(self, raw_documents):
        embeddings = np.zeros((len(raw_documents), self.vector_size))

        for i, doc in enumerate(raw_documents):
            tokens = self.preprocess(doc)
            N_w = len(tokens)
            doc_emb = np.zeros((1, self.vector_size))

            for token in tokens:
                if token in self.word2vec_model:
                    vec = self.word2vec_model[token].reshape(1, -1)
                else:
                    vec = np.zeros((1, self.vector_size))
                doc_emb += vec

            if N_w == 0:
                N_w = 1  # Avoid division by zero
            embeddings[i] = doc_emb / N_w

        return embeddings

    def fit(self, X, y=None):
        return self


In [ ]:
import pickle

# Define file names based on timestamp
word2vec_results_filename = "/content/drive/MyDrive/Colab_Objects/output/word2vec_df_results_2025-07-17_12-31-47.pkl"
word2vec_best_model_filename = "/content/drive/MyDrive/Colab_Objects/output/word2vec_best_per_model_2025-07-17_12-31-47.pkl"
word2vec_external_metrics_filename = "/content/drive/MyDrive/Colab_Objects/output/word2vec_external_metrics_2025-07-17_12-31-47.pkl"

# Load df_results
with open(word2vec_results_filename, 'rb') as f:
    word2vec_df_results = pickle.load(f)

# Load best_per_model
with open(word2vec_best_model_filename, 'rb') as f:
    word2vec_best_per_model = pickle.load(f)

# Load df_external_metrics
with open(word2vec_external_metrics_filename, 'rb') as f:
    word2vec_df_external_metrics = pickle.load(f)

# Display the loaded objects
print("\n======= [Word2Vec] Final Averaged Results Across All Folds =======")
print(word2vec_df_results)

print("\n======= [Word2Vec] External Test Set Metrics =======")
print(word2vec_df_external_metrics)

print("\n[Word2Vec] Best fold per classifier (F1-score):")
for name, entry in word2vec_best_per_model.items():
    print(f"{name}: {entry}")



======= [Word2Vec] Final Averaged Results Across All Folds =======
   Accuracy  Precision    Recall  F1-Score                Model
0  0.835135   0.586239  0.577897  0.567499  Logistic Regression
1  0.933703   0.929825  0.842314  0.861684           Linear SVM
2  0.912162   0.923074  0.796476  0.821867        Random Forest
3  0.808140   0.654612  0.651603  0.632719        Decision Tree
4  0.955164   0.931784  0.912277  0.911808       MLP Classifier

======= [Word2Vec] External Test Set Metrics =======
                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.444444   0.425397  0.436601  0.374691
1           Linear SVM  0.638889   0.676282  0.652288  0.640810
2        Random Forest  0.333333   0.476716  0.397386  0.313921
3        Decision Tree  0.222222   0.253247  0.216807  0.171252
4       MLP Classifier  0.750000   0.720238  0.783007  0.711691

[Word2Vec] Best fold per classifier (F1-score):
Logistic Regression: {'f1': 0.6070074852683549, 'model': Logi

In [ ]:
word2vec_best_per_model

{'Logistic Regression': {'f1': 0.6070074852683549,
  'model': LogisticRegression(max_iter=1000),
  'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 0x7d2f3f6b0c90>)},
 'Linear SVM': {'f1': 0.9105782564843159,
  'model': LinearSVC(),
  'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 0x7d2f3f6b0c90>)},
 'Random Forest': {'f1': 0.8651915027691425,
  'model': RandomForestClassifier(random_state=42),
  'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 0x7d2f3f6b0c90>)},
 'Decision Tree': {'f1': 0.699625928055914,
  'model': GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
               param_grid={'max_depth': [8, 12, None],
                           'min_samples_leaf': [1, 2, 3]},
               scoring='f1_macro'),
  'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors obj

<h3>SBERT learner</h3>

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import nltk
import re

nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

class BERTVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="all-MiniLM-L6-v2", max_chunk_chars=300):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.max_chunk_chars = max_chunk_chars  # Roughly ~100–150 words

    def split_into_chunks(self, text):
        # Split into sentences
        sentences = sent_tokenize(text)
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) < self.max_chunk_chars:
                current_chunk += " " + sentence
            else:
                chunks.append(current_chunk.strip())
                current_chunk = sentence

        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks

    def transform(self, raw_documents):
        final_embeddings = []

        for doc in raw_documents:
            chunks = self.split_into_chunks(doc)
            chunk_embeddings = self.model.encode(chunks, convert_to_numpy=True, show_progress_bar=False)

            # Average embeddings across all chunks
            doc_embedding = np.mean(chunk_embeddings, axis=0)
            final_embeddings.append(doc_embedding)

        return np.array(final_embeddings)

    def fit(self, X, y=None):
        return self


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
import pickle

# Define file names based on timestamp
sbert_results_filename = "/content/drive/MyDrive/Colab_Objects/output/sbert_df_results_2025-07-16_18-26-47.pkl"
sbert_best_model_filename = "/content/drive/MyDrive/Colab_Objects/output/sbert_best_per_model_2025-07-16_18-26-47.pkl"
sbert_external_metrics_filename = "/content/drive/MyDrive/Colab_Objects/output/sbert_external_metrics_2025-07-16_18-26-47.pkl"

# Load df_results
with open(sbert_results_filename, 'rb') as f:
    sbert_df_results = pickle.load(f)

# Load best_per_model
with open(sbert_best_model_filename, 'rb') as f:
    sbert_best_per_model = pickle.load(f)

# Load df_external_metrics
with open(sbert_external_metrics_filename, 'rb') as f:
    sbert_df_external_metrics = pickle.load(f)

# Display the loaded objects
print("\n======= [SBERT] Final Averaged Results Across All Folds =======")
print(sbert_df_results)

print("\n======= [SBERT] External Test Set Metrics =======")
print(sbert_df_external_metrics)

print("\n[SBERT] Best fold per classifier (F1-score):")
for name, entry in sbert_best_per_model.items():
    print(f"{name}: {entry}")



======= [SBERT] Final Averaged Results Across All Folds =======
   Accuracy  Precision    Recall  F1-Score                Model
0  0.849421   0.629952  0.600014  0.598989  Logistic Regression
1  0.937194   0.912258  0.855770  0.868262           Linear SVM
2  0.919273   0.906171  0.817499  0.838763        Random Forest
3  0.759685   0.613207  0.608059  0.596369        Decision Tree
4  0.967728   0.948922  0.945086  0.942515       MLP Classifier

======= [SBERT] External Test Set Metrics =======
                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.555556   0.569153  0.522876  0.472135
1           Linear SVM  0.694444   0.677778  0.728758  0.665709
2        Random Forest  0.416667   0.314134  0.340850  0.294931
3        Decision Tree  0.361111   0.220408  0.219608  0.203271
4       MLP Classifier  0.722222   0.753535  0.692810  0.638038

[SBERT] Best fold per classifier (F1-score):
Logistic Regression: {'f1': 0.643059163059163, 'model': LogisticRegres

<h3> Ensemble Learning</h3>

In [ ]:
tfidf_best_per_model.items()

dict_items([('Logistic Regression', {'f1': 0.8425555859943715, 'model': LogisticRegression(max_iter=1000), 'vectoriser': TfidfVectorizer()}), ('Linear SVM', {'f1': 0.9595345027643163, 'model': SVC(kernel='linear', probability=True), 'vectoriser': TfidfVectorizer()}), ('Random Forest', {'f1': 0.9338345864661654, 'model': RandomForestClassifier(random_state=42), 'vectoriser': TfidfVectorizer()}), ('Decision Tree', {'f1': 0.9329004329004329, 'model': GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [8, 12, None],
                         'min_samples_leaf': [1, 2, 3]},
             scoring='f1_macro'), 'vectoriser': TfidfVectorizer()}), ('MLP Classifier', {'f1': 0.9807938407655523, 'model': GridSearchCV(cv=3, estimator=MLPClassifier(max_iter=500, random_state=42),
             n_jobs=-1,
             param_grid={'hidden_layer_sizes': [(100, 80), (120, 100, 80, 50),
                                                (100, 8

In [ ]:
word2vec_best_per_model.items()

dict_items([('Logistic Regression', {'f1': 0.6070074852683549, 'model': LogisticRegression(max_iter=1000), 'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 0x7d2f3f6b0c90>)}), ('Linear SVM', {'f1': 0.9105782564843159, 'model': LinearSVC(), 'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 0x7d2f3f6b0c90>)}), ('Random Forest', {'f1': 0.8651915027691425, 'model': RandomForestClassifier(random_state=42), 'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 0x7d2f3f6b0c90>)}), ('Decision Tree', {'f1': 0.699625928055914, 'model': GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [8, 12, None],
                         'min_samples_leaf': [1, 2, 3]},
             scoring='f1_macro'), 'vectoriser': Word2VecVectorizer(word2vec_model=<gensim.models.keyedvectors.KeyedVectors object at 

In [ ]:

tfidf_mi = 4 #MLP
word2vec_mi = 4 #MLP
sbert_mi = 4  #MLP

tfidf_model = list(tfidf_best_per_model.items())[tfidf_mi][1]['model']
tfidf_vec = list(tfidf_best_per_model.items())[tfidf_mi][1]['vectoriser']

word2vec_model = list(word2vec_best_per_model.items())[word2vec_mi][1]['model']
word2vec_vec = list(word2vec_best_per_model.items())[word2vec_mi][1]['vectoriser']

sbert_model = list(sbert_best_per_model.items())[sbert_mi][1]['model']
sbert_vec = list(sbert_best_per_model.items())[sbert_mi][1]['vectoriser']

<h3>External dataset</h3>

In [ ]:
user_df = pd.read_csv('/content/drive/MyDrive/Colab_Objects/data/user_test_survey.csv')
user_df.head(3)

,Timestamp,Which domain of study are you currently pursuing or have you completed?,Describe in a few paragraphs why you are/were interested in this field? (at least 100 words please).,"Which specific sub-domain within your field of interest would you like to pursue or have you pursued? For example, Electrical Engineering or Safety Management.",Any reasons why? At least 50 words please,What is your education level?
0,2024/12/03 4:17:47 pm EET,Human Sciences,I have always enjoyed learning about culture a...,I would like to purse a course in Psychology,To understand human behaviour and mental healt...,"Postgraduate level (Masters, Doctorate)"
1,2024/12/03 4:38:30 pm EET,"Social Work, Employment Relations and Human Re...",I am committed to making a meaningful impact o...,Research,This is a new and exciting career opportunity ...,"Postgraduate level (Masters, Doctorate)"
2,2024/12/03 4:47:56 pm EET,Education,Education is the driving point towards a bette...,I have Bed Degree Intermediate & Senior phase ...,Content is Data the ways in which is why it is...,"University graduate (Diploma, Advanced diploma..."


In [ ]:
test_df = user_df.iloc[:,[2,1]]
test_df.rename(columns={test_df.columns[0]: 'features', test_df.columns[1]: 'target'}, inplace=True)
test_df = test_df[test_df['target'].isin(y.unique())]#get only qualified targets
test_df['features'] = test_df['features'].apply(preprocess)
test_df.head()

,features,target
0,always enjoyed learning culture communication ...,Human Sciences
2,education driving point towards better south a...,Education
4,always interested psychopathology particularly...,Human Sciences
5,maintain environmental sustainability djdjdjdn...,Agriculture and Environmental Sciences
6,interested field information technology gives ...,"Science, Engineering and Technology"


In [ ]:
X_ext = test_df['features']
y_ext = test_df['target']

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import pandas as pd

# === Assume you have pipelines like this defined already ===
pipe_1 = Pipeline([('tfidf', tfidf_vec), ('tfidf_model', tfidf_model)])
pipe_2 = Pipeline([('word2vec', word2vec_vec), ('word2vec_model', word2vec_model)])
pipe_3 = Pipeline([('sbert_vec', sbert_vec), ('sbert_model', sbert_model)])

# Define base learners (each is a pipeline: vectoriser + learner)
base_learners = [
    ('tfidf_model_pipe', pipe_1),
    ('word2vec_model_pipe', pipe_2),
    ('sbert_model_pipe', pipe_3)
]

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt_parameters = {'max_depth':[None, 8, 10], 'min_samples_leaf':[1,2,3]}
dt = DecisionTreeClassifier()
dt_clf = GridSearchCV(dt, dt_parameters,cv=5)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# === 1. Already trained base learners ===
trained_base_learners = {
    'tfidf_model_pipe': pipe_1,
    'word2vec_model_pipe': pipe_2,
    'sbert_model_pipe': pipe_3
}

# === 2. Voting weights ===
voting_weights = {
    'tfidf_model_pipe': 1,
    'word2vec_model_pipe': 5,
    'sbert_model_pipe': 2
}

# === 3. Evaluation helper ===
def eval_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

# === 4. Base learners' metrics ===
train_results = []
test_results = []

for name, model in trained_base_learners.items():
    train_preds = model.predict(X)
    test_preds = model.predict(X_ext)

    train_metrics = eval_metrics(y, train_preds)
    test_metrics = eval_metrics(y_ext, test_preds)

    train_results.append({'Model': name, 'Type': 'Base', **train_metrics})
    test_results.append({'Model': name, 'Type': 'Base', **test_metrics})

# === 5. Get predicted probabilities ===
probs_train = {k: m.predict_proba(X) for k, m in trained_base_learners.items()}
probs_test = {k: m.predict_proba(X_ext) for k, m in trained_base_learners.items()}

# === 6. Weighted soft voting ===
def soft_vote(probs_dict, weights_dict):
    total_weight = sum(weights_dict.values())
    weighted_sum = sum(weights_dict[k] * probs_dict[k] for k in probs_dict)
    return weighted_sum / total_weight

voting_probs_train = soft_vote(probs_train, voting_weights)
voting_probs_test  = soft_vote(probs_test, voting_weights)

voting_preds_train = np.argmax(voting_probs_train, axis=1)
voting_preds_test  = np.argmax(voting_probs_test, axis=1)

# Decode predictions if needed (match y label type)
if not np.issubdtype(y.dtype, np.integer):
    classes = trained_base_learners['tfidf_model_pipe'].classes_
    voting_preds_train = classes[voting_preds_train]
    voting_preds_test  = classes[voting_preds_test]

# === 7. Manual stacking ===
X_meta_train = np.column_stack([probs_train[k][:, 1] for k in trained_base_learners])
X_meta_test  = np.column_stack([probs_test[k][:, 1] for k in trained_base_learners])

meta_learner = dt_clf
meta_learner.fit(X_meta_train, y)

stacking_preds_train = meta_learner.predict(X_meta_train)
stacking_preds_test  = meta_learner.predict(X_meta_test)

# === 8. Ensemble metrics ===
for name, preds_train, preds_test in [
    ('VotingClassifier', voting_preds_train, voting_preds_test),
    ('StackingClassifier', stacking_preds_train, stacking_preds_test)
]:
    train_metrics = eval_metrics(y, preds_train)
    test_metrics = eval_metrics(y_ext, preds_test)

    train_results.append({'Model': name, 'Type': 'Ensemble', **train_metrics})
    test_results.append({'Model': name, 'Type': 'Ensemble', **test_metrics})

# === 9. Format Outputs ===
train_results_df = pd.DataFrame(train_results).sort_values(by=['Type', 'Model'])
test_results_df = pd.DataFrame(test_results).sort_values(by=['Type', 'Model'])

# Ensure 'Base' models show first
type_order = ['Base', 'Ensemble']
train_results_df['Type'] = pd.Categorical(train_results_df['Type'], categories=type_order, ordered=True)
test_results_df['Type'] = pd.Categorical(test_results_df['Type'], categories=type_order, ordered=True)
train_results_df = train_results_df.sort_values(by=['Type', 'Model'])
test_results_df = test_results_df.sort_values(by=['Type', 'Model'])

# === 10. Display ===
print("\nTrain Set Performance:")
print(train_results_df.to_string(index=False))

print("\nTest Set Performance:")
print(test_results_df.to_string(index=False))



Train Set Performance:
              Model     Type  Accuracy  Precision   Recall       F1
   sbert_model_pipe     Base  0.994624   0.994917 0.994624 0.994659
   tfidf_model_pipe     Base  0.996416   0.996423 0.996416 0.996409
word2vec_model_pipe     Base  0.996416   0.996539 0.996416 0.996437
 StackingClassifier Ensemble  0.933692   0.939531 0.933692 0.932914
   VotingClassifier Ensemble  0.996416   0.996539 0.996416 0.996437

Test Set Performance:
              Model     Type  Accuracy  Precision   Recall       F1
   sbert_model_pipe     Base  0.722222   0.855724 0.722222 0.724440
   tfidf_model_pipe     Base  0.472222   0.628704 0.472222 0.503961
word2vec_model_pipe     Base  0.750000   0.827183 0.750000 0.765329
 StackingClassifier Ensemble  0.194444   0.292824 0.194444 0.185269
   VotingClassifier Ensemble  0.777778   0.839683 0.777778 0.789138
